# Import Required Libraries
Import necessary libraries including pandas, numpy, scikit-learn, and natural language processing tools.

In [1]:
# Import necessary libraries
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations
from sklearn.feature_extraction.text import CountVectorizer  # For text vectorization
from sklearn.cluster import KMeans  # For clustering transactions
from nltk.corpus import stopwords  # For natural language processing
from nltk.tokenize import word_tokenize  # For tokenizing text
import nltk

# Download NLTK data (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/mains/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/mains/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load and Explore Transactions Data
Load the transactions dataset, examine its structure, check for missing values, and perform basic exploratory data analysis to understand transaction patterns.

In [2]:
# Load the transactions dataset
transactions = pd.read_csv('transactions.csv')  # Replace with the actual file path

# Display the first few rows of the dataset
print("First few rows of the dataset:")
print(transactions.head())

# Check the structure of the dataset
print("\nDataset Info:")
transactions.info()

# Check for missing values
print("\nMissing Values:")
print(transactions.isnull().sum())

# Perform basic exploratory data analysis
# Summary statistics for numerical columns
print("\nSummary Statistics:")
print(transactions.describe())

# Check the distribution of transaction amounts
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.hist(transactions['Amount'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Transaction Amounts')
plt.xlabel('Transaction Amount ($)')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

# Check the number of unique merchants
print("\nNumber of unique merchants:")
print(transactions['Merchant'].nunique())

# Display the most frequent merchants
print("\nTop 10 most frequent merchants:")
print(transactions['Merchant'].value_counts().head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'transactions.csv'

# Preprocess Transaction Descriptions
Clean and normalize transaction descriptions by removing special characters, converting to lowercase, tokenizing, and removing stopwords to prepare for categorization.

In [ ]:
# Clean and normalize transaction descriptions

# Define a function to preprocess transaction descriptions
def preprocess_description(description):
    # Convert to lowercase
    description = description.lower()
    # Remove special characters and numbers
    description = ''.join(char for char in description if char.isalpha() or char.isspace())
    # Tokenize the text
    tokens = word_tokenize(description)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Join tokens back into a single string
    return ' '.join(tokens)

# Apply the preprocessing function to the 'Description' column
transactions['Cleaned_Description'] = transactions['Description'].apply(preprocess_description)

# Display the first few rows of the updated dataset
print("\nFirst few rows after preprocessing descriptions:")
print(transactions[['Description', 'Cleaned_Description']].head())

# Feature Engineering for Categorization
Extract features from transaction data including merchant names, transaction amounts, keywords in descriptions, and temporal patterns that might indicate specific categories.

In [ ]:
# Extract features from transaction data

# Feature 1: Extract merchant names
transactions['Merchant_Name'] = transactions['Merchant']

# Feature 2: Transaction amount bins
bins = [-np.inf, 50, 100, 500, 1000, np.inf]
labels = ['Low', 'Medium-Low', 'Medium', 'Medium-High', 'High']
transactions['Amount_Bin'] = pd.cut(transactions['Amount'], bins=bins, labels=labels)

# Feature 3: Keywords in descriptions
vectorizer = CountVectorizer(max_features=50, stop_words='english')
description_features = vectorizer.fit_transform(transactions['Cleaned_Description'])
description_features_df = pd.DataFrame(description_features.toarray(), columns=vectorizer.get_feature_names_out())
transactions = pd.concat([transactions, description_features_df], axis=1)

# Feature 4: Temporal patterns (e.g., day of the week, hour of the day)
transactions['Transaction_Date'] = pd.to_datetime(transactions['Date'])  # Ensure 'Date' is in datetime format
transactions['Day_of_Week'] = transactions['Transaction_Date'].dt.day_name()
transactions['Hour_of_Day'] = transactions['Transaction_Date'].dt.hour

# Display the first few rows of the dataset with new features
print("\nFirst few rows after feature engineering:")
print(transactions.head())

# Build a Rule-based Categorization System
Develop a rule-based system using regex patterns and keyword matching to categorize common transaction types based on description text and other transaction attributes.

In [ ]:
# Define a function for rule-based categorization
def categorize_transaction(description):
    """
    Categorize transactions based on description using regex patterns and keyword matching.
    """
    import re

    # Define rules for categorization
    rules = {
        'Groceries': r'\b(grocery|supermarket|store|mart)\b',
        'Dining': r'\b(restaurant|cafe|dining|food|eatery)\b',
        'Transportation': r'\b(taxi|uber|lyft|bus|train|metro|transport)\b',
        'Entertainment': r'\b(cinema|movie|theater|concert|entertainment|show)\b',
        'Utilities': r'\b(electricity|water|gas|utility|bill|internet)\b',
        'Shopping': r'\b(shop|mall|retail|clothing|apparel|fashion)\b',
        'Healthcare': r'\b(hospital|clinic|pharmacy|health|medical|doctor)\b',
        'Travel': r'\b(hotel|flight|airline|travel|vacation|trip)\b',
        'Education': r'\b(school|college|university|education|course|tuition)\b',
        'Miscellaneous': r'\b(other|misc|general|unknown)\b'
    }

    # Check each rule and assign category
    for category, pattern in rules.items():
        if re.search(pattern, description, re.IGNORECASE):
            return category

    # Default category if no match is found
    return 'Uncategorized'

# Apply the categorization function to the 'Cleaned_Description' column
transactions['Category'] = transactions['Cleaned_Description'].apply(categorize_transaction)

# Display the first few rows of the dataset with categorized transactions
print("\nFirst few rows after categorizing transactions:")
print(transactions[['Description', 'Cleaned_Description', 'Category']].head())

# Visualize the distribution of categorized transactions
plt.figure(figsize=(10, 6))
category_counts = transactions['Category'].value_counts()
sns.barplot(x=category_counts.index, y=category_counts.values, palette='viridis')
plt.title('Distribution of Categorized Transactions')
plt.xlabel('Category')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Implement Machine Learning Categorization
Create a machine learning pipeline to categorize transactions using techniques like TF-IDF vectorization and classification algorithms (Random Forest, SVM, or Naive Bayes).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Prepare the data for machine learning
# Use 'Cleaned_Description' as features and 'Category' as labels
X = transactions['Cleaned_Description']
y = transactions['Category']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train a Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test_tfidf)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

# Apply Categorization to All Transactions
Apply the developed categorization methods to the entire dataset, combining rule-based and ML approaches for optimal results.

In [ ]:
# Apply the trained model and rule-based categorization to all transactions

# Define a function to combine rule-based and ML-based categorization
def combined_categorization(description):
    """
    Apply rule-based categorization first. If the transaction is still uncategorized,
    use the machine learning model for categorization.
    """
    # Apply rule-based categorization
    category = categorize_transaction(description)
    
    # If rule-based categorization fails, use the ML model
    if category == 'Uncategorized':
        # Transform the description using the TF-IDF vectorizer
        description_tfidf = tfidf_vectorizer.transform([description])
        # Predict the category using the trained Random Forest model
        category = rf_classifier.predict(description_tfidf)[0]
    
    return category

# Apply the combined categorization function to all transactions
transactions['Final_Category'] = transactions['Cleaned_Description'].apply(combined_categorization)

# Display the first few rows of the dataset with final categorized transactions
print("\nFirst few rows after applying combined categorization:")
print(transactions[['Description', 'Cleaned_Description', 'Final_Category']].head())

# Visualize the distribution of final categorized transactions
plt.figure(figsize=(10, 6))
final_category_counts = transactions['Final_Category'].value_counts()
sns.barplot(x=final_category_counts.index, y=final_category_counts.values, palette='viridis')
plt.title('Distribution of Final Categorized Transactions')
plt.xlabel('Category')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Evaluate Categorization Performance
Assess the performance of the categorization system using metrics like accuracy, precision, recall, and F1-score, with visualizations of the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

# Evaluate the performance of the categorization system
# Generate a classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

# Visualize the confusion matrix
plt.figure(figsize=(12, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=rf_classifier.classes_)
disp.plot(cmap='Blues', xticks_rotation=45, ax=plt.gca())
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Additional metrics visualization (optional)
# Convert confusion matrix to a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.tight_layout()
plt.show()